In [1]:
import os, sys, json
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT  = r'C:\Users\chiku\OneDrive\Documents\ecom_dynamic_pricing_optimization'
PROCESSED_DIR = os.path.join(PROJECT_ROOT, 'data', 'processed')

# Load data and results
train = pd.read_parquet(os.path.join(PROCESSED_DIR, 'train_with_cate.parquet'))

with open(os.path.join(PROCESSED_DIR, 'dml_results.json')) as f:
    dml_results = json.load(f)

with open(os.path.join(PROCESSED_DIR, 'col_definitions.json')) as f:
    cols = json.load(f)

ate = dml_results['ate']

print(f'Loaded: {train.shape}')
print(f'ATE elasticity: {ate:.4f}')
print(f'CATE range: [{dml_results["cate_min"]:.4f}, {dml_results["cate_max"]:.4f}]')

Loaded: (500000, 21)
ATE elasticity: -0.0829
CATE range: [-1.0367, 0.4643]


In [2]:
# ─── Revenue-Maximising Price ─────────────────────────────────────────────────
# Economics: Profit maximisation requires MR = MC
# In log-log demand model: Q = A * P^ε
# Revenue R = P * Q = A * P^(1+ε)
# dR/dP = A * (1+ε) * P^ε = 0
# → Revenue maximised when ε = -1 (unit elastic)
# → If ε > -1 (inelastic): raise price to increase revenue
# → If ε < -1 (elastic):   lower price to increase revenue
#
# Optimal price given elasticity ε and cost c:
# Lerner condition: (P-MC)/P = -1/ε
# → P* = MC * ε / (ε + 1)      [only valid when ε < -1]
# → When -1 < ε < 0 (inelastic): P* → ∞, so we use guardrail cap

def optimal_price(current_price, elasticity, unit_cost,
                  max_change_pct=0.30, min_margin_pct=0.10):
    """
    Compute revenue-maximising price subject to guardrails.
    
    Uses Lerner condition: P* = MC * ε / (ε + 1)
    Falls back to guardrail upper bound when demand is inelastic.
    """
    # Lerner optimal price (valid only when ε < -1)
    if elasticity < -1:
        p_star = unit_cost * elasticity / (elasticity + 1)
    else:
        # Inelastic demand → raise price as much as guardrail allows
        p_star = current_price * (1 + max_change_pct)

    # Guardrail 1: max ±30% change from current price
    p_star = np.clip(p_star, 
                     current_price * (1 - max_change_pct),
                     current_price * (1 + max_change_pct))

    # Guardrail 2: minimum margin floor
    margin_floor = unit_cost / (1 - min_margin_pct)
    p_star = max(p_star, margin_floor)

    return round(p_star, 2)

# ─── Apply to product sample ──────────────────────────────────────────────────
# Get one row per product with its CATE elasticity
product_sample = (
    train.groupby('product_id')
    .agg(
        current_price=('price', 'mean'),
        elasticity=('estimated_elasticity', 'mean'),
        units_sold=('add_to_cart_order', 'sum'),
        department=('department', 'first'),
    )
    .reset_index()
)

# Simulate unit cost as 40% of current price (typical grocery margin ~60%)
product_sample['unit_cost'] = product_sample['current_price'] * 0.40

# Compute optimal price for each product
product_sample['optimal_price'] = product_sample.apply(
    lambda r: optimal_price(r['current_price'], r['elasticity'], r['unit_cost']),
    axis=1
)

product_sample['price_change_pct'] = (
    (product_sample['optimal_price'] - product_sample['current_price'])
    / product_sample['current_price']
)

product_sample['revenue_current']  = product_sample['current_price'] * product_sample['units_sold']
product_sample['revenue_optimal']  = product_sample['optimal_price'] * product_sample['units_sold']
product_sample['revenue_lift_pct'] = (
    (product_sample['revenue_optimal'] - product_sample['revenue_current'])
    / product_sample['revenue_current']
)

print('=' * 55)
print('PRICING OPTIMIZER RESULTS')
print('=' * 55)
print(f'\nProducts analysed:        {len(product_sample):,}')
print(f'ATE elasticity used:      {ate:.4f}')
print(f'\nPrice change distribution:')
print(f'  Mean change:            {product_sample["price_change_pct"].mean():.1%}')
print(f'  % getting price increase: {(product_sample["price_change_pct"] > 0).mean():.1%}')
print(f'  % getting price decrease: {(product_sample["price_change_pct"] < 0).mean():.1%}')
print(f'  % at guardrail upper:   {(product_sample["price_change_pct"] >= 0.299).mean():.1%}')
print(f'\nRevenue impact:')
print(f'  Total current revenue:  ${product_sample["revenue_current"].sum():>12,.0f}')
print(f'  Total optimal revenue:  ${product_sample["revenue_optimal"].sum():>12,.0f}')
print(f'  Overall revenue lift:   {(product_sample["revenue_optimal"].sum() / product_sample["revenue_current"].sum() - 1):.1%}')
print(f'\nTop 5 products by revenue lift:')
print(product_sample.nlargest(5, 'revenue_lift_pct')[
    ['product_id', 'department', 'current_price', 'optimal_price', 
     'price_change_pct', 'revenue_lift_pct']
].to_string(index=False))

PRICING OPTIMIZER RESULTS

Products analysed:        29,094
ATE elasticity used:      -0.0829

Price change distribution:
  Mean change:            30.0%
  % getting price increase: 100.0%
  % getting price decrease: 0.0%
  % at guardrail upper:   84.2%

Revenue impact:
  Total current revenue:  $  16,682,183
  Total optimal revenue:  $  21,687,050
  Overall revenue lift:   30.0%

Top 5 products by revenue lift:
 product_id    department  current_price  optimal_price  price_change_pct  revenue_lift_pct
      43324        snacks       0.511667           0.67          0.309446          0.309446
       8181        snacks       0.550000           0.72          0.309091          0.309091
      46293       missing       0.550000           0.72          0.309091          0.309091
      34659 personal care       0.550000           0.72          0.309091          0.309091
       1592 personal care       0.573333           0.75          0.308140          0.308140


In [ ]:
# ─── Plot price change distribution by department ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Price change distribution
axes[0].hist(product_sample['price_change_pct'] * 100, bins=30, 
             color='#2563EB', alpha=0.8, edgecolor='white')
axes[0].axvline(0, color='red', linestyle='--', lw=2, label='No change')
axes[0].set_xlabel('Recommended Price Change (%)')
axes[0].set_ylabel('Number of Products')
axes[0].set_title('Distribution of Price Recommendations\n(Inelastic demand → all products get increase)')
axes[0].legend()

# Revenue lift by department
dept_revenue = (
    product_sample.groupby('department')
    .agg(
        current=('revenue_current', 'sum'),
        optimal=('revenue_optimal', 'sum')
    )
    .assign(lift=lambda x: (x['optimal'] / x['current'] - 1) * 100)
    .sort_values('lift')
)

axes[1].barh(dept_revenue.index, dept_revenue['lift'],
             color='#16A34A', alpha=0.8)
axes[1].axvline(0, color='black', lw=0.5)
axes[1].set_xlabel('Revenue Lift (%)')
axes[1].set_title('Revenue Lift by Department\n(Guardrail-constrained)')

plt.tight_layout()
plt.savefig(os.path.join(PROCESSED_DIR, 'pricing_optimizer_results.png'), 
            dpi=150, bbox_inches='tight')
plt.show()

# ─── Save results ─────────────────────────────────────────────────────────────
product_sample.to_parquet(os.path.join(PROCESSED_DIR, 'pricing_recommendations.parquet'), 
                          index=False)

optimizer_summary = {
    'n_products': int(len(product_sample)),
    'pct_price_increase': float((product_sample['price_change_pct'] > 0).mean()),
    'pct_price_decrease': float((product_sample['price_change_pct'] < 0).mean()),
    'mean_price_change_pct': float(product_sample['price_change_pct'].mean()),
    'total_revenue_current': float(product_sample['revenue_current'].sum()),
    'total_revenue_optimal': float(product_sample['revenue_optimal'].sum()),
    'overall_revenue_lift_pct': float(
        product_sample['revenue_optimal'].sum() / 
        product_sample['revenue_current'].sum() - 1
    ),
}
with open(os.path.join(PROCESSED_DIR, 'optimizer_summary.json'), 'w') as f:
    json.dump(optimizer_summary, f, indent=2)

print('✅ Pricing recommendations saved → data/processed/pricing_recommendations.parquet')
print('✅ Optimizer summary saved       → data/processed/optimizer_summary.json')
print('✅ Plot saved                    → data/processed/pricing_optimizer_results.png')
print('\nNotebook 07 complete!')
print('\n' + '=' * 55)
print('PROJECT SUMMARY')
print('=' * 55)
print(f'  Naive OLS elasticity:   {dml_results["naive_ols_elasticity"] if "naive_ols_elasticity" in dml_results else "N/A"}')
print(f'  DML elasticity (ATE):   {ate:.4f}')
print(f'  Endogeneity bias:        {dml_results["endogeneity_bias"]:.4f}')
print(f'  Products priced:         {len(product_sample):,}')
print(f'  Revenue lift:            {optimizer_summary["overall_revenue_lift_pct"]:.1%}')
print('=' * 55)

✅ Pricing recommendations saved → data/processed/pricing_recommendations.parquet
✅ Optimizer summary saved       → data/processed/optimizer_summary.json
✅ Plot saved                    → data/processed/pricing_optimizer_results.png

Notebook 07 complete!

PROJECT SUMMARY
  Naive OLS elasticity:   N/A
  DML elasticity (ATE):   -0.0829
  Endogeneity bias:        0.0874
  Products priced:         29,094
  Revenue lift:            30.0%


: 